# Day 2: Logistic Regression and Binary Classification

## Introduction

In the previous notebook, we studied **Linear Regression**, where the goal was to predict a continuous numerical value.

In this notebook, we move from regression to **classification**. Classification is a supervised machine learning task in which the model predicts a category or class instead of a continuous number.

We will focus on **binary classification**, where the target variable has only two possible classes, usually represented as:

* `0`: Negative class
* `1`: Positive class

Examples of binary classification problems include:

* Predicting whether an email is spam or not spam.
* Predicting whether a customer will leave a service or remain.
* Predicting whether a medical test result indicates a disease or no disease.
* Predicting whether a financial transaction is fraudulent or legitimate.
  
During this notebook, we will build a complete binary classification workflow using Logistic Regression and Scikit-learn. We will inspect and clean the data, encode the binary target, select appropriate features, create a stratified train-test split, preprocess missing and categorical values, train baseline and class-balanced models, generate probabilities and class predictions, and evaluate the models using appropriate classification metrics.



## Regression vs. Classification

Supervised machine learning problems can generally be divided into two major categories:

### Regression

Regression models predict a continuous numerical value.

Examples include predicting:

* House prices
* Monthly sales
* Temperature
* Delivery time

### Classification

Classification models predict a discrete class or category.

Examples include predicting:

* Spam or not spam
* Fraud or legitimate transaction
* Customer churn or customer retention
* Disease or no disease

The correct modelling and evaluation methods depend on the type of target variable. Therefore, identifying whether a problem is a regression problem or a classification problem is one of the first decisions in a machine learning workflow.


## How Logistic Regression Works

Despite its name, Logistic Regression is a classification algorithm rather than a regression algorithm.

The model first calculates a linear score from the input features:

$$
z = b + w_1x_1 + w_2x_2 + \cdots + w_mx_m
$$

where:

* $x_1, x_2, \ldots, x_m$ are the input features
* $w_1, w_2, \ldots, w_m$ are the learned feature coefficients
* $b$ is the intercept or bias
* $z$ is the resulting linear score

A raw linear score can be any value from negative infinity to positive infinity. It therefore cannot be interpreted directly as a probability.

Logistic Regression passes this score through the sigmoid function:

$$
P(y=1 \mid X) = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

The sigmoid function transforms any linear score into a value between `0` and `1`. This value is interpreted as the estimated probability that the observation belongs to the positive class.

For example:

* A probability of `0.10` indicates low estimated confidence in the positive class.
* A probability of `0.50` lies at the default decision boundary.
* A probability of `0.90` indicates high estimated confidence in the positive class.

Using the default threshold of `0.5`, the class prediction is:

$$
\hat{y} =
\begin{cases}
1, & \text{if } P(y=1 \mid X) \geq 0.5, \\
0, & \text{if } P(y=1 \mid X) < 0.5.
\end{cases}
$$

The coefficients determine how each feature changes the log-odds of the positive class:

$$
\log\left(\frac{p}{1-p}\right) = b + w_1x_1 + \cdots + w_mx_m
$$

A positive coefficient increases the estimated odds of class `1`, while a negative coefficient decreases them. Exponentiating a coefficient produces its odds ratio:

$$
\text{Odds Ratio} = e^w
$$

An odds ratio greater than `1` is associated with higher estimated odds of the positive class, while an odds ratio below `1` is associated with lower estimated odds.

These coefficients describe associations learned from the data. They do not establish that a feature causes the target outcome.

## Understanding Features and the Target Variable

A supervised machine learning dataset contains two main components:

* **Features (`X`)**: The input variables used by the model to make predictions.
* **Target (`y`)**: The correct output that the model is expected to learn and predict.

Each row in `X` represents one data sample, while each column represents one feature.

If a dataset contains $n$ samples and $m$ features, the feature matrix can be represented as:

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1m} \\
x_{21} & x_{22} & \cdots & x_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{nm}
\end{bmatrix}
$$

The corresponding target vector is:

$$
$$
y =
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_n
\end{bmatrix}
$$
$$

For binary classification, every target value belongs to one of two classes:

$$
\{0, 1\}
$$

The feature matrix and target vector must contain the same number of samples:

$$
\text{Number of rows in } X = \text{Number of values in } y
$$

The model learns a relationship between the features in `X` and the known class labels in `y`.


## Loading a Real COVID-19 Testing Dataset

Instead of using a synthetic dataset, we will work with a real historical COVID-19 testing dataset.

The dataset contains anonymized records of individuals who were tested for COVID-19. Each row represents one tested individual, while the columns describe recorded symptoms, demographic information, the reason for testing, and the actual test result.

The dataset contains the following variables:

* `test_date`: The date of the COVID-19 test
* `cough`: Whether the individual reported coughing
* `fever`: Whether the individual reported fever
* `sore_throat`: Whether the individual reported a sore throat
* `shortness_of_breath`: Whether the individual reported difficulty breathing
* `head_ache`: Whether the individual reported a headache
* `age_60_and_above`: Whether the individual was aged 60 or older
* `gender`: The recorded gender
* `test_indication`: The reason or indication for performing the test
* `corona_result`: The result of the COVID-19 test

Our machine learning target will be `corona_result`.

After cleaning the target variable, the classification task will contain two classes:

* `negative`: The test did not detect COVID-19
* `positive`: The test detected COVID-19

Therefore, this is a binary classification problem.



In [ ]:
from pathlib import Path
import pandas as pd

dataset_path = Path(
    "../corona dataset/"
    "corona_tested_individuals_ver_006.english.csv"
)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"The dataset was not found at: {dataset_path.resolve()}"
    )

df = pd.read_csv(
    dataset_path,
    low_memory=False
)

print("Dataset loaded successfully.")
print("Dataset path:", dataset_path)
print("Dataset shape:", df.shape)

df.head()

## Initial Dataset Inspection

Before cleaning, transforming, or modelling the dataset, we must inspect its original structure.

The initial inspection will help us answer the following questions:

- How many rows and columns does the dataset contain?
- What are the column names?
- What data type was assigned to each column?
- Which columns contain missing values?
- What values appear in the target variable?
- How many observations belong to each target class?

At this stage, we will not modify the dataset. We will first understand the raw data and identify the preprocessing decisions required before modelling.

In [ ]:
print("Dataset shape:")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst five rows:")
display(df.head())

In [ ]:
print("Column data types:")
print(df.dtypes)

In [ ]:
missing_count = df.isna().sum()
missing_percentage = (
    df.isna().mean() * 100
).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})
missing_summary

In [ ]:
target_values = df["corona_result"].unique()
target_counts = df["corona_result"].value_counts()
target_percentages =df["corona_result"].value_counts(normalize=True).mul(100).round(2)


print("Unique target values:")
print(target_values)

print("\nTarget value counts:")
print(target_counts)

print("\nTarget value percentages:")
print(target_percentages)

### Initial Inspection Findings

The dataset contains **278,848 observations** and **10 columns**.

The initial inspection revealed the following:

- The symptom columns are mostly binary, with values representing the presence or absence of each symptom.
- Some symptom columns contain a small number of missing values.
- The `age_60_and_above` column has a substantial amount of missing data.
- The `gender` column also contains missing values, although at a lower rate.
- The `corona_result` target does not contain missing values.
- The raw target contains three values: `negative`, `positive`, and `other`.
- The target is strongly imbalanced, with the majority of observations belonging to the `negative` class.

The value `other` is a recorded target category, not a missing value. Since our objective is binary classification, records labelled as `other` will require separate handling before modelling.

The class imbalance also means that accuracy alone may provide a misleading evaluation of model performance. Additional classification metrics will be required later in the notebook.

## Cleaning and Encoding the Target Variable

The raw target column, `corona_result`, contains three recorded values:

- `negative`
- `positive`
- `other`

Our objective is to build a binary classification model that distinguishes between negative and positive COVID-19 test results.

The value `other` is not a missing value. It is an explicitly recorded category whose meaning does not correspond clearly to either the negative or positive class. Assigning it to one of the two classes would introduce an unsupported label into the data.

Therefore, observations labelled as `other` will be excluded from the binary classification dataset.

After filtering the target, the remaining labels will be encoded numerically as:

- `negative` $\rightarrow 0$
- `positive` $\rightarrow 1$

The original DataFrame will remain unchanged. A separate DataFrame named `df_binary` will be created for the binary classification workflow.

In [ ]:
valid_target_labels = ["negative", "positive"]

binary_target_mask = df["corona_result"].isin(valid_target_labels)

df_binary = df.loc[binary_target_mask].copy().reset_index(drop=True)


rows_before_filtering = len(df)
rows_after_filtering = len(df_binary)
rows_removed = rows_before_filtering - rows_after_filtering

print("Rows before filtering:", rows_before_filtering)
print("Rows after filtering:", rows_after_filtering)
print("Rows removed:", rows_removed)

print("\nRemaining target values:")
print(df_binary["corona_result"].unique())

In [ ]:
target_counts = df_binary["corona_result"].value_counts().sort_index()

target_percentages = df_binary["corona_result"].value_counts(normalize=True).sort_index().mul(100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

target_summary.index.name = "corona_result"

display(target_summary)

### Target Cleaning Results

The original dataset contained **278,848 observations**.

A total of **3,892 observations** labelled as `other` were excluded because they did not belong clearly to either class in the binary classification task.

The resulting binary classification dataset contains **274,956 observations** and only two target labels:

* `negative`
* `positive`

The filtered target contains no missing values.

The class distribution remains strongly imbalanced:

* Approximately **94.64%** of the observations belong to the negative class.
* Approximately **5.36%** belong to the positive class.

At this stage, the target labels are still stored as text. In the next step, they will be encoded numerically as:

* `negative` $\rightarrow 0$
* `positive` $\rightarrow 1$

This imbalance will later be considered during data splitting and model evaluation. Accuracy alone will not be sufficient to judge the model's ability to identify positive cases.


In [ ]:
target_mapping = {
    "negative": 0,
    "positive": 1
}

df_binary["target"] = (
    df_binary["corona_result"]
    .map(target_mapping)
    .astype("int8")
)

assert df_binary["target"].isna().sum() == 0
assert set(df_binary["target"].unique()) == {0, 1}

print("Target encoding completed successfully.")

print("\nTarget label mapping:")
display(df_binary[["corona_result", "target"]].drop_duplicates().sort_values("target").reset_index(drop=True))

print("\nEncoded target distribution:")
display(df_binary["target"].value_counts().sort_index().rename_axis("target").to_frame("count"))

### Target Encoding Results

The binary target was encoded successfully in a new column named `target`.

The encoding is:

- `negative` $\rightarrow 0$
- `positive` $\rightarrow 1$

The original `corona_result` column was preserved for readability, while the numerical `target` column will be used during model training.

The encoded target contains no missing values and only the expected values `0` and `1`.

The class distribution remains strongly imbalanced:

- Class `0` contains 260,227 observations.
- Class `1` contains 14,729 observations.

This imbalance must be preserved during the train-test split using stratified sampling and considered when selecting evaluation metrics.

## Selecting the Model Features

The dataset contains several columns that could potentially be used as model inputs.

For the initial baseline model, we will use the recorded symptoms and categorical patient information:

### Symptom Features

- `cough`
- `fever`
- `sore_throat`
- `shortness_of_breath`
- `head_ache`

### Categorical Features

- `age_60_and_above`
- `gender`
- `test_indication`

The `corona_result` column will not be included because it is the original target label.

The `target` column will also not be included in the feature matrix because it represents the output the model must predict.

The `test_date` column will be excluded from the initial baseline model. A raw date string cannot be used directly by Logistic Regression, and including the date without careful temporal analysis may cause the model to learn changes in testing patterns over time instead of learning relationships between patient characteristics and the test result.

A raw feature matrix named `X_raw` and a target vector named `y` will now be created. The name `X_raw` indicates that the features have not yet been imputed or encoded.

In [ ]:
symptom_features = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache"
]

categorical_features = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

feature_columns = symptom_features + categorical_features

X_raw = df_binary[feature_columns].copy()
y = df_binary["target"].copy()

assert len(X_raw) == len(y)
assert X_raw.index.equals(y.index)
assert "corona_result" not in X_raw.columns
assert "target" not in X_raw.columns

print("Feature matrix and target vector created successfully.")
print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)

print("\nSelected features:")
print(X_raw.columns.tolist())

print("\nFirst five feature rows:")
display(X_raw.head())

print("\nFirst five target values:")
display(y.head())

## Inspecting the Selected Features

Before splitting or preprocessing the data, we must inspect the selected features carefully.

This inspection will identify:

- The data type of each feature
- The number and percentage of missing values
- The number of unique recorded values
- The frequency of every value in each feature
- Whether the symptom columns contain only the expected binary values
- Which categorical values must be encoded before modelling

Missing values will be included during value inspection rather than silently ignored.

No missing-value imputation or categorical encoding will be performed at this stage. These preprocessing operations must later be learned from the training data only to prevent data leakage.

In [ ]:
feature_summary = pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing_count": X_raw.isna().sum(),
    "missing_percentage": (
        X_raw.isna().mean() * 100
    ).round(2),
    "unique_values_including_missing": (
        X_raw.nunique(dropna=False)
    )
})

display(feature_summary)

In [ ]:
def summarize_feature_values(data, column):
    value_summary = (
        data[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )

    value_summary["percentage"] = (
        value_summary["count"]
        .div(len(data))
        .mul(100)
        .round(2)
    )

    return value_summary


for column in X_raw.columns:
    print(f"\n{'=' * 60}")
    print(f"Feature: {column}")
    print(f"{'=' * 60}")

    display(
        summarize_feature_values(
            data=X_raw,
            column=column
        )
    )

### Feature Inspection Strategy

The selected features include two different data types that will require separate preprocessing strategies.

The symptom features are numerical binary variables. Their expected values are `0` and `1`, although some observations may contain missing values.

The patient-information features are categorical variables. Their text categories cannot be passed directly to Logistic Regression and will require categorical encoding.

The missing values will not be removed or filled using the complete dataset. Instead, preprocessing will be fitted only on the training data after the train-test split.

The raw feature matrix will therefore remain unchanged until the preprocessing pipeline is constructed.

### Feature Inspection Findings

The feature inspection confirmed that the selected columns contain only the expected recorded values.

The five symptom features are numerical binary variables containing `0` and `1`. Their missing-value rates are very small:

* `cough` and `fever` each contain 252 missing values, representing approximately 0.09% of the dataset.
* `sore_throat`, `shortness_of_breath`, and `head_ache` each contain only one missing value.

The symptom frequencies also differ considerably. Cough and fever are recorded more frequently than sore throat, shortness of breath, and headache.

The categorical features require additional preprocessing:

* `age_60_and_above` contains 125,664 missing values, representing approximately 45.70% of the observations.
* `gender` contains 19,045 missing values, representing approximately 6.93% of the observations.
* `test_indication` contains no missing values and has three categories: `Other`, `Abroad`, and `Contact with confirmed`.

The large amount of missing data in `age_60_and_above` should not be ignored. Removing every row with a missing age-group value would discard almost half of the binary classification dataset. Therefore, missing categorical values will later be represented as a separate category rather than deleting those observations.

No preprocessing will be fitted before splitting the data. The dataset will first be divided into training and testing sets, after which missing-value handling and categorical encoding will be learned from the training data only.


## Stratified Train-Test Split

Before preprocessing or training the model, the dataset must be divided into separate training and testing sets.

The training set will be used to:

* Learn how missing values should be handled
* Learn the categorical encoding
* Estimate the Logistic Regression coefficients

The testing set will remain unseen during training and will be used only to evaluate how well the completed pipeline generalizes to new observations.

An 80/20 split will be used:

* 80% of the observations will form the training set.
* 20% will form the testing set.

Because the target is strongly imbalanced, the split will use stratification. Stratification preserves approximately the same positive and negative class percentages in the full dataset, training set, and testing set.

A fixed `random_state` will also be used to make the split reproducible.


In [ ]:
from sklearn.model_selection import train_test_split


X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


assert len(X_train_raw) == len(y_train)
assert len(X_test_raw) == len(y_test)

assert X_train_raw.index.equals(y_train.index)
assert X_test_raw.index.equals(y_test.index)

assert set(X_train_raw.index).isdisjoint(
    set(X_test_raw.index)
)


print("Train-test split completed successfully.")

print("\nDataset sizes:")
print(f"Full dataset:  {len(X_raw):,} observations")
print(f"Training set:  {len(X_train_raw):,} observations")
print(f"Testing set:   {len(X_test_raw):,} observations")


split_class_distribution = pd.DataFrame({
    "full_dataset": (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    ),
    "training_set": (
        y_train.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    ),
    "testing_set": (
        y_test.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )
}).round(2)

split_class_distribution.index.name = "target"

print("\nClass percentages after splitting:")
display(split_class_distribution)

## Building the Preprocessing and Modelling Pipeline

The raw feature matrix cannot be passed directly to Logistic Regression because it contains missing values and text-based categorical features.

Two separate preprocessing strategies will be used.

### Symptom Features

The symptom columns are binary numerical features containing `0` and `1`.

Their small number of missing values will be replaced with the most frequently recorded value in each symptom column.

### Categorical Features

The categorical columns contain text labels that Logistic Regression cannot process directly.

Missing categorical values will first be replaced with an explicit category named `Missing`. This is particularly important for `age_60_and_above`, where a large proportion of observations have no recorded value.

The categorical features will then be transformed using one-hot encoding. One-hot encoding creates numerical indicator columns for the recorded categories.

The preprocessing steps and Logistic Regression model will be combined inside a single Scikit-learn `Pipeline`.

Using a pipeline ensures that:

* Preprocessing is learned from the training data only.
* The same transformations are applied consistently to the testing data.
* The complete workflow can be fitted and used as one reproducible object.
* Data leakage caused by preprocessing the full dataset is prevented.

The initial model will be an unweighted baseline Logistic Regression model. It will not yet apply class balancing, allowing us to observe how the standard model behaves on the imbalanced target.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


symptom_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)


categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "symptoms",
            symptom_preprocessor,
            symptom_features
        ),
        (
            "categorical",
            categorical_preprocessor,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


baseline_model.fit(
    X_train_raw,
    y_train
)


transformed_feature_names = (
    baseline_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


print("Baseline Logistic Regression trained successfully.")

print(
    "\nNumber of features before preprocessing:",
    X_train_raw.shape[1]
)

print(
    "Number of features after preprocessing:",
    len(transformed_feature_names)
)

print("\nTransformed feature names:")
print(transformed_feature_names)

## Evaluating the Baseline Logistic Regression Model

The trained pipeline will now generate two types of output for the testing set:

* **Class predictions:** the final predicted class, either `0` or `1`
* **Positive-class probabilities:** the estimated probability that each observation belongs to class `1`

By default, Logistic Regression predicts the positive class when its estimated probability is at least `0.5`.

Because the target is strongly imbalanced, the model will be evaluated using several complementary metrics:

* **Accuracy:** the proportion of all predictions that are correct
* **Balanced accuracy:** the average recall across both classes
* **Precision:** among observations predicted as positive, the proportion that are actually positive
* **Recall:** among all actual positive observations, the proportion correctly identified
* **F1-score:** the harmonic balance between precision and recall
* **ROC-AUC:** the model's ability to rank positive observations above negative observations across different classification thresholds
* **Confusion matrix:** the counts of correct and incorrect predictions for each class

Accuracy must not be interpreted alone. A model that predicts nearly every observation as negative could achieve high accuracy because the negative class forms the large majority of the dataset, while still failing to identify many positive observations.


In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


y_pred_baseline = baseline_model.predict(
    X_test_raw
)

y_probability_baseline = baseline_model.predict_proba(
    X_test_raw
)[:, 1]


baseline_metrics = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "ROC-AUC"
        ],
        "score": [
            accuracy_score(
                y_test,
                y_pred_baseline
            ),
            balanced_accuracy_score(
                y_test,
                y_pred_baseline
            ),
            precision_score(
                y_test,
                y_pred_baseline,
                zero_division=0
            ),
            recall_score(
                y_test,
                y_pred_baseline,
                zero_division=0
            ),
            f1_score(
                y_test,
                y_pred_baseline,
                zero_division=0
            ),
            roc_auc_score(
                y_test,
                y_probability_baseline
            )
        ]
    }
)

baseline_metrics["score"] = (
    baseline_metrics["score"]
    .round(4)
)


print("Baseline evaluation metrics:")
display(baseline_metrics)


confusion_matrix_values = confusion_matrix(
    y_test,
    y_pred_baseline
)

confusion_matrix_table = pd.DataFrame(
    confusion_matrix_values,
    index=[
        "Actual Negative (0)",
        "Actual Positive (1)"
    ],
    columns=[
        "Predicted Negative (0)",
        "Predicted Positive (1)"
    ]
)

print("\nConfusion matrix:")
display(confusion_matrix_table)


print("\nDetailed classification report:")
print(
    classification_report(
        y_test,
        y_pred_baseline,
        target_names=[
            "Negative (0)",
            "Positive (1)"
        ],
        digits=4,
        zero_division=0
    )
)


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_baseline,
    display_labels=[
        "Negative (0)",
        "Positive (1)"
    ],
    values_format=",d"
)

plt.title(
    "Baseline Logistic Regression Confusion Matrix"
)
plt.tight_layout()
plt.show()

### Baseline Model Results

The baseline Logistic Regression model achieved the following results on the unseen testing set:

* **Accuracy:** 0.9638
* **Balanced Accuracy:** 0.7468
* **Precision:** 0.7379
* **Recall:** 0.5037
* **F1-score:** 0.5987
* **ROC-AUC:** 0.8903

The confusion matrix shows:

* **51,519 true negatives:** negative observations correctly classified as negative
* **527 false positives:** negative observations incorrectly classified as positive
* **1,462 false negatives:** positive observations incorrectly classified as negative
* **1,484 true positives:** positive observations correctly classified as positive

Although the model achieved an accuracy of 96.38%, this result must be interpreted carefully because approximately 94.64% of the observations belong to the negative class.

The positive-class recall is only 50.37%. Therefore, the baseline model fails to identify almost half of the actual positive observations.

Its positive-class precision is 73.79%, meaning that approximately three out of every four positive predictions are correct.

The ROC-AUC score of 0.8903 indicates that the model has a relatively strong ability to rank positive observations above negative observations. However, the default probability threshold of 0.5 does not produce sufficiently high positive-class recall.

A class-balanced Logistic Regression model will now be trained to place greater importance on the minority positive class.


## Training a Class-Balanced Logistic Regression Model

The baseline model treats every training observation equally. Because the negative class is much larger than the positive class, minimizing the overall classification error can cause the model to focus primarily on the majority negative class.

A second Logistic Regression model will therefore be trained using:

```python
class_weight="balanced"
```

This setting automatically assigns a larger weight to observations from the minority class and a smaller weight to observations from the majority class.

The weights are calculated from the class frequencies in the training data. The less frequent positive observations will have a greater influence on the model's training objective.

The purpose is not to artificially create additional observations. Instead, misclassifying a positive observation will become more costly during model training.

The expected trade-off is:

* Higher positive-class recall
* Fewer false negatives
* More false positives
* Possibly lower overall accuracy and precision

The balanced model will use exactly the same training and testing sets as the baseline model, allowing a fair comparison.


In [ ]:
from sklearn.base import clone


balanced_model = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


balanced_model.fit(
    X_train_raw,
    y_train
)


print(
    "Class-balanced Logistic Regression "
    "trained successfully."
)

print(
    "\nClass weight setting:",
    balanced_model
    .named_steps["classifier"]
    .class_weight
)

## Evaluating the Class-Balanced Model

The class-balanced model will be evaluated on the same untouched testing set using the same metrics applied to the baseline model.

Using identical evaluation data and metrics ensures that differences in performance result from the class-weighting strategy rather than from a different data split.

Particular attention will be given to:

* Positive-class recall
* Positive-class precision
* F1-score
* Balanced accuracy
* The number of false negatives and false positives

The balanced model will be considered useful if it identifies substantially more positive observations without producing an unacceptable number of false positive predictions.


In [ ]:
y_pred_balanced = balanced_model.predict(
    X_test_raw
)

y_probability_balanced = balanced_model.predict_proba(
    X_test_raw
)[:, 1]


balanced_metrics = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "ROC-AUC"
        ],
        "score": [
            accuracy_score(
                y_test,
                y_pred_balanced
            ),
            balanced_accuracy_score(
                y_test,
                y_pred_balanced
            ),
            precision_score(
                y_test,
                y_pred_balanced,
                zero_division=0
            ),
            recall_score(
                y_test,
                y_pred_balanced,
                zero_division=0
            ),
            f1_score(
                y_test,
                y_pred_balanced,
                zero_division=0
            ),
            roc_auc_score(
                y_test,
                y_probability_balanced
            )
        ]
    }
)

balanced_metrics["score"] = (
    balanced_metrics["score"]
    .round(4)
)


print("Balanced-model evaluation metrics:")
display(balanced_metrics)


balanced_confusion_matrix = confusion_matrix(
    y_test,
    y_pred_balanced
)

balanced_confusion_matrix_table = pd.DataFrame(
    balanced_confusion_matrix,
    index=[
        "Actual Negative (0)",
        "Actual Positive (1)"
    ],
    columns=[
        "Predicted Negative (0)",
        "Predicted Positive (1)"
    ]
)

print("\nBalanced-model confusion matrix:")
display(balanced_confusion_matrix_table)


print("\nDetailed classification report:")
print(
    classification_report(
        y_test,
        y_pred_balanced,
        target_names=[
            "Negative (0)",
            "Positive (1)"
        ],
        digits=4,
        zero_division=0
    )
)


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_balanced,
    display_labels=[
        "Negative (0)",
        "Positive (1)"
    ],
    values_format=",d"
)

plt.title(
    "Class-Balanced Logistic Regression "
    "Confusion Matrix"
)
plt.tight_layout()
plt.show()

In [ ]:
model_comparison = (
    baseline_metrics
    .rename(columns={"score": "baseline"})
    .merge(
        balanced_metrics.rename(
            columns={"score": "balanced"}
        ),
        on="metric"
    )
)

model_comparison["difference"] = (
    model_comparison["balanced"]
    - model_comparison["baseline"]
).round(4)


print("Baseline vs. balanced model:")
display(model_comparison)

## Interpreting the Logistic Regression Coefficients

Logistic Regression is an interpretable linear classification model. Each transformed feature receives a learned coefficient.

The coefficient sign indicates the direction of the relationship:

* A positive coefficient is associated with higher estimated log-odds of the positive class.
* A negative coefficient is associated with lower estimated log-odds of the positive class.
* A coefficient near zero indicates a relatively weak contribution after accounting for the other model features.

The exponentiated coefficient is called the odds ratio:

* An odds ratio greater than `1` indicates higher estimated odds.
* An odds ratio below `1` indicates lower estimated odds.
* An odds ratio close to `1` indicates little change in estimated odds.

For binary symptom features, each coefficient compares the presence of the symptom (`1`) with its absence (`0`).

For one-hot encoded categorical features, each coefficient compares the displayed category with the category removed by `drop="first"`, known as the reference category.

The baseline model will be used for coefficient interpretation because it represents the standard, unweighted Logistic Regression fit.


In [ ]:
import numpy as np


baseline_classifier = (
    baseline_model
    .named_steps["classifier"]
)

baseline_intercept = (
    baseline_classifier.intercept_[0]
)

baseline_coefficients = (
    baseline_classifier.coef_[0]
)


coefficient_table = pd.DataFrame({
    "feature": transformed_feature_names,
    "coefficient": baseline_coefficients,
    "odds_ratio": np.exp(baseline_coefficients),
    "absolute_coefficient": np.abs(
        baseline_coefficients
    )
})


coefficient_table = (
    coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "Baseline-model intercept:",
    round(baseline_intercept, 4)
)

print("\nCoefficients ordered by absolute size:")
display(
    coefficient_table[
        [
            "feature",
            "coefficient",
            "odds_ratio"
        ]
    ].round(4)
)

In [ ]:
fitted_encoder = (
    baseline_model
    .named_steps["preprocessor"]
    .named_transformers_["categorical"]
    .named_steps["one_hot_encoder"]
)


reference_categories = []

for feature, categories, dropped_index in zip(
    categorical_features,
    fitted_encoder.categories_,
    fitted_encoder.drop_idx_
):
    reference_categories.append({
        "categorical_feature": feature,
        "reference_category": categories[
            dropped_index
        ]
    })


reference_category_table = pd.DataFrame(
    reference_categories
)


print(
    "Reference categories removed by "
    "drop='first':"
)

display(reference_category_table)

In [ ]:
coefficient_plot_data = (
    coefficient_table
    .sort_values("coefficient")
)


plt.figure(figsize=(10, 7))

plt.barh(
    coefficient_plot_data["feature"],
    coefficient_plot_data["coefficient"]
)

plt.axvline(
    x=0,
    linewidth=1
)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Transformed Feature")

plt.title(
    "Baseline Logistic Regression "
    "Feature Coefficients"
)

plt.tight_layout()
plt.show()

## Final Model Comparison and Conclusion

Two Logistic Regression models were trained and evaluated on the same stratified testing set.

### Baseline Model

The baseline model achieved:

* Accuracy: **96.38%**
* Balanced accuracy: **74.68%**
* Positive-class precision: **73.79%**
* Positive-class recall: **50.37%**
* Positive-class F1-score: **59.87%**
* ROC-AUC: **89.03%**

It correctly identified 1,484 positive observations but incorrectly classified 1,462 positive observations as negative.

The baseline model therefore produced relatively reliable positive predictions, but it failed to identify almost half of the actual positive observations.

### Class-Balanced Model

The class-balanced model achieved:

* Accuracy: **92.76%**
* Balanced accuracy: **83.88%**
* Positive-class precision: **40.39%**
* Positive-class recall: **73.93%**
* Positive-class F1-score: **52.24%**
* ROC-AUC: **89.35%**

It correctly identified 2,178 positive observations and missed 768 positive observations.

Compared with the baseline model, class weighting:

* Increased positive-class recall by **23.56 percentage points**
* Increased balanced accuracy by **9.20 percentage points**
* Reduced false negatives from **1,462 to 768**
* Prevented **694 additional positive observations** from being missed
* Increased false positives from **527 to 3,215**
* Reduced positive-class precision from **73.79% to 40.39%**
* Reduced the positive-class F1-score from **59.87% to 52.24%**

The ROC-AUC scores are very similar. This indicates that both models have similar overall ranking ability, while class weighting mainly changes how strongly the model prioritizes the minority positive class.

### Model Selection

There is no universally best model. The appropriate choice depends on the cost of each type of error.

The baseline model is preferable when false positive predictions are particularly costly and higher precision is required.

The class-balanced model is preferable for a screening-oriented objective in which failing to identify a positive observation is considered more costly than generating additional false alerts.

For this educational screening task, the class-balanced model is selected as the more suitable candidate because it substantially improves positive-class recall and reduces the number of missed positive observations. However, its lower precision means that many predicted positive observations would require further verification.


## Day 2 Completion Summary

This notebook completed an end-to-end binary classification workflow:

1. Distinguished classification from regression.
2. Defined the feature matrix and binary target.
3. Loaded and inspected a real historical dataset.
4. Cleaned and numerically encoded the target.
5. Examined feature values and missing data.
6. Selected appropriate model inputs.
7. Created a reproducible stratified train-test split.
8. Built leakage-safe numerical and categorical preprocessing.
9. Used one-hot encoding for categorical features.
10. Combined preprocessing and modelling in a Scikit-learn pipeline.
11. Trained a baseline Logistic Regression model.
12. Generated class predictions and positive-class probabilities.
13. Evaluated the model using multiple classification metrics.
14. Interpreted the confusion matrix and class imbalance.
15. Trained and evaluated a class-balanced model.
16. Compared the trade-off between precision and recall.
17. Interpreted Logistic Regression coefficients and odds ratios.
18. Documented the model choice, and key findings.
    
The next stage of the internship can build on this notebook through broader model-evaluation techniques such as cross-validation, threshold analysis, hyperparameter tuning, and comparison with additional classification algorithms.
